# Direction Policy Kaggle Pipeline

This notebook prepares raw M5 forex CSV data, generates direction labels, and trains/replays one or more of the five neural architectures.

The first code cell now works whether Kaggle leaves your project as a `.zip` file or automatically extracts it into an input dataset folder.

Edit `RAW_DATASET_DIR`, and optionally set `PROJECT_INPUT` to your Kaggle project dataset folder if auto-detection does not find it.


In [ ]:
import os
from pathlib import Path

print("Exists:", os.path.exists("/kaggle/input"))
print("Contents of /kaggle/input:")
!ls -lah /kaggle/input

print("\nDirectory tree:")
!find /kaggle/input -maxdepth 6 -print | head -200

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

# === EDIT THESE ===
# This can point to either:
#   1) a Kaggle input folder containing the unzipped project, e.g. /kaggle/input/direction-policy-kaggle-ready
#   2) a direct zip file, e.g. /kaggle/input/direction-policy-kaggle-ready/direction_policy_kaggle_ready.zip
#   3) None, in which case the notebook scans /kaggle/input and tries to find the project automatically.
PROJECT_INPUT = None  # e.g. '/kaggle/input/direction-policy-kaggle-ready'
RAW_DATASET_DIR = '/kaggle/input/datasets/chrisrose100/mt5-dataset/raw'
SYMBOLS = ['EURUSD']
SYMBOLS2 =  'EURUSD'
TRAIN_START = '2024-01-01'
TRAIN_END = '2025-03-01'
REPLAY_START = '2025-03-01'
REPLAY_END = '2025-06-01'
EPOCHS = 50
BATCH_SIZE = 512
DEVICE = 'cuda'  # use 'cpu' if no GPU accelerator is enabled
RESET_WORKDIR = True

WORKDIR = Path('/kaggle/working/project')
WORKDIR.parent.mkdir(parents=True, exist_ok=True)


def is_project_root(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / 'kaggle_run_pipeline.py').exists()
        and (p / 'requirements_kaggle.txt').exists()
        and (p / 'src').is_dir()
        and (p / 'config').is_dir()
    )


def find_unzipped_project_root(base: Path) -> Path | None:
    """Find the project root when Kaggle has already unzipped the uploaded project dataset."""
    if is_project_root(base):
        return base
    if not base.exists() or not base.is_dir():
        return None

    # Search shallow first because Kaggle datasets often extract files directly under the dataset folder.
    for child in base.iterdir():
        if is_project_root(child):
            return child

    # Fallback recursive search, kept bounded enough for normal Kaggle datasets.
    for p in base.rglob('kaggle_run_pipeline.py'):
        root = p.parent
        if is_project_root(root):
            return root
    return None


def find_project_zip(base: Path) -> Path | None:
    if base.is_file() and base.suffix.lower() == '.zip':
        return base
    if base.exists() and base.is_dir():
        preferred = list(base.glob('direction_policy_kaggle_ready*.zip'))
        if preferred:
            return preferred[0]
        zips = list(base.glob('*.zip'))
        if zips:
            return zips[0]
    return None


def autodetect_project_input() -> Path:
    kaggle_input = Path('/kaggle/input')
    candidates = []
    if PROJECT_INPUT:
        candidates.append(Path(PROJECT_INPUT))
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.iterdir() if p.exists()])

    # Prefer an already-unzipped project root.
    for c in candidates:
        root = find_unzipped_project_root(c)
        if root is not None:
            print('Detected unzipped project root:', root)
            return root

    # Fallback to a zip file.
    for c in candidates:
        z = find_project_zip(c)
        if z is not None:
            print('Detected project zip:', z)
            return z

    checked = '\n'.join(str(c) for c in candidates)
    raise FileNotFoundError(
        'Could not find the Kaggle-ready project. Set PROJECT_INPUT to your project dataset folder or zip.\n'
        f'Checked:\n{checked}'
    )


project_source = autodetect_project_input()

if RESET_WORKDIR and WORKDIR.exists():
    shutil.rmtree(WORKDIR)

if project_source.is_file() and project_source.suffix.lower() == '.zip':
    WORKDIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(project_source, 'r') as zf:
        zf.extractall(WORKDIR)
else:
    shutil.copytree(
        project_source,
        WORKDIR,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '.git', 'models', 'logs')
    )

os.chdir(WORKDIR)
print('Project directory:', Path.cwd())
print('Raw dataset directory:', RAW_DATASET_DIR)
print('Available configs:')
for p in sorted(Path('config').glob('direction_settings*.yaml')):
    print(' -', p)


In [ ]:
# Install Kaggle-safe requirements. MetaTrader5 is intentionally not installed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements_kaggle.txt'], check=True)


In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print("GPU:", name)
    print("CUDA capability:", capability)

    if capability[0] < 7:
        print("WARNING: This GPU is likely incompatible with the installed PyTorch CUDA build.")
        DEVICE = "cpu"
    else:
        DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("Using DEVICE =", DEVICE)

In [ ]:
!python -m src.kaggle_prepare_raw_data --input-dir /kaggle/input/datasets/chrisrose100/mt5-dataset/raw --output-dir data/raw --timeframe M5 --symbols $SYMBOLS2

In [ ]:
!python -m src.prepare_mt5_data \
  --config config/direction_settings_residual_mlp.yaml \
  --symbols $SYMBOLS2 

In [ ]:
!python -m src.prepare_direction_dataset \
  --config config/direction_settings_residual_mlp.yaml \
  --workers 2 \
  --date-start 2024-01-01 \
  --date-end 2025-06-01 \
  --out-dir data/direction \
  --symbols $SYMBOLS2

In [ ]:
!python /kaggle/working/project/kaggle_run_pipeline_all_symbols.py \
  --input-data-root $RAW_DATASET_DIR \
  --timeframe M5 \
  --mode train-all \
  --skip-direction-prep \
  --skip-raw-copy \
  --skip-direction-prep \
  --train-start 2024-01-01 \
  --train-end 2025-03-01 \
  --replay-start 2025-03-01 \
  --replay-end 2025-06-01 \
  --device cuda \
  --epochs 50 \
  --batch-size 512 \
  --configs all \
  --sides buy sell \
  --symbols $SYMBOLS2

In [ ]:
#%cd /kaggle/working
#%rm hazard_logs.zip
#%rm hazards_results.zip
#%cd /kaggle/working/project
#%ls
#%rm -r /kaggle/working/project/data/processed_m5

In [ ]:
 %cd /kaggle/working

!zip -r logs.zip /kaggle/working/project/logs/
!zip -r models.zip /kaggle/working/project/models/
from IPython.display import FileLink
FileLink(r'logs.zip')
FileLink(r'models.zip')
